# Embed All Benchmark Items

This notebook embeds all ~103K benchmark items from `items.parquet` using `nomic-ai/nomic-embed-text-v1.5`.

**Requirements:** Run on Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU).

**Estimated runtime:** ~3-5 minutes for embedding on a T4 GPU.

## Cell 1: Install dependencies

In [ ]:
!pip install -q sentence-transformers pandas pyarrow

## Cell 2: Upload or load data

Choose **one** of the two options below.

In [ ]:
# === Option A: Upload items.parquet via Colab file upload ===
from google.colab import files
uploaded = files.upload()  # Select items.parquet when prompted
PARQUET_PATH = 'items.parquet'
print(f"Uploaded: {list(uploaded.keys())}")

In [ ]:
# === Option B: Load from Google Drive (uncomment and run instead of Option A) ===
# from google.colab import drive
# drive.mount('/content/drive')
# PARQUET_PATH = '/content/drive/MyDrive/path/to/items.parquet'  # <-- adjust path

## Cell 3: Load and inspect data

In [ ]:
import pandas as pd

df = pd.read_parquet(PARQUET_PATH)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nNull content: {df['content'].isna().sum()}")

print(f"\n--- Benchmark distribution ---")
print(df['benchmark_id'].value_counts().to_string())

content_lens = df['content'].str.len()
print(f"\n--- Content length stats ---")
print(f"  Median: {content_lens.median():.0f} chars")
print(f"  P90:    {content_lens.quantile(0.9):.0f} chars")
print(f"  Max:    {content_lens.max():.0f} chars")
print(f"  >2048:  {(content_lens > 2048).sum()} items ({(content_lens > 2048).mean()*100:.1f}%)")

print(f"\n--- Sample items ---")
for _, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[{row['benchmark_id']}] {row['item_id']}")
    print(f"  content: {row['content'][:200]}{'...' if len(row['content']) > 200 else ''}")

## Cell 4: Load embedding model

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# Check GPU
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {gpu_mem:.1f} GB")
    device = 'cuda'
else:
    print("WARNING: No GPU detected! Embedding will be very slow.")
    print("Go to Runtime -> Change runtime type -> T4 GPU")
    device = 'cpu'

# Load model in float16 to save GPU memory
MODEL_NAME = 'nomic-ai/nomic-embed-text-v1.5'
try:
    model = SentenceTransformer(
        MODEL_NAME,
        trust_remote_code=True,
        device=device,
        model_kwargs={"torch_dtype": torch.float16},
    )
    EMBEDDING_DIM = 768
    print(f"Loaded {MODEL_NAME} (dim={EMBEDDING_DIM}, fp16)")
except Exception as e:
    print(f"Failed to load {MODEL_NAME}: {e}")
    print("Falling back to BAAI/bge-large-en-v1.5...")
    MODEL_NAME = 'BAAI/bge-large-en-v1.5'
    model = SentenceTransformer(MODEL_NAME, device=device)
    EMBEDDING_DIM = 1024
    print(f"Loaded {MODEL_NAME} (dim={EMBEDDING_DIM})")

if device == 'cuda':
    alloc = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used by model: {alloc:.2f} GB")

## Cell 5: Embed all items

In [ ]:
import time
import numpy as np

# Prepare texts with the appropriate prefix
contents = df['content'].tolist()
if 'nomic' in MODEL_NAME:
    # nomic-embed-text requires "search_query: " prefix
    texts = [f"search_query: {c}" for c in contents]
else:
    texts = contents

# Sort by length so similar-length texts are batched together.
# This prevents one long text from forcing an entire batch to be padded to 8K tokens.
sort_idx = np.argsort([len(t) for t in texts])
texts_sorted = [texts[i] for i in sort_idx]

print(f"Embedding {len(texts_sorted)} items (sorted by length for memory efficiency)...")
start = time.time()

embeddings_sorted = model.encode(
    texts_sorted,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,  # L2 normalize to unit length
    convert_to_numpy=True,
    precision="float32",  # output precision (model runs in fp16 internally)
)

# Unsort back to original order
unsort_idx = np.argsort(sort_idx)
embeddings = embeddings_sorted[unsort_idx]

elapsed = time.time() - start
print(f"\nDone in {elapsed:.1f}s ({len(texts)/elapsed:.0f} items/sec)")
print(f"Embeddings shape: {embeddings.shape}, dtype: {embeddings.dtype}")

# Verify normalization
norms = np.linalg.norm(embeddings, axis=1)
print(f"L2 norms — min: {norms.min():.6f}, max: {norms.max():.6f}, mean: {norms.mean():.6f}")

# Free GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after encoding: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## Cell 6: Save results

In [ ]:
import json

# Save embeddings
np.save('item_embeddings.npy', embeddings.astype(np.float32))
print(f"Saved item_embeddings.npy ({embeddings.nbytes / 1e6:.0f} MB)")

# Save metadata
meta = {
    'item_ids': df['item_id'].tolist(),
    'benchmark_ids': df['benchmark_id'].tolist(),
    'model_name': MODEL_NAME,
    'embedding_dim': EMBEDDING_DIM,
    'num_items': len(df),
}
with open('item_embedding_meta.json', 'w') as f:
    json.dump(meta, f)
print(f"Saved item_embedding_meta.json ({meta['num_items']} items, dim={meta['embedding_dim']})")

## Cell 7: Download results

In [ ]:
from google.colab import files
files.download('item_embeddings.npy')
files.download('item_embedding_meta.json')